In [411]:
import torch

batch_size = 32
device = torch.device("cuda")

device

device(type='cuda')

In [404]:
import torchvision.datasets as datasets
import torchvision.transforms as T
from torch.utils.data import DataLoader

train_transform = T.Compose([
    T.Resize((64, 64)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    T.ToTensor(),
])

test_transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
])

train_dataset = datasets.ImageFolder(root='train_data', transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size)

test_dataset = datasets.ImageFolder(root='test_data', transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

train_dataset[0][0][0][0][0]

tensor(0.1412)

In [405]:
import torchvision.models as models

model = torch.load('vgg16.pth', weights_only=False)
model.fc = torch.nn.Identity()
model = model.to(device)

model.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Conv2d(128, 256

In [406]:
import numpy as np
X = []

with torch.no_grad():
    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        X.append(outputs.cpu().numpy())

X = np.concatenate(X, axis=0)
X.shape

(201, 1000)

In [407]:
X[:3]

array([[-0.5721163 , -2.587995  ,  1.5226755 , ..., -6.6458635 ,
         2.0695672 ,  6.6387515 ],
       [-5.0567474 , -1.5285807 , -4.157093  , ..., -6.835312  ,
         1.1585841 ,  8.701349  ],
       [-1.359276  ,  0.87372637, -1.0230285 , ..., -7.0119095 ,
         0.52183634, 11.48093   ]], dtype=float32)

In [408]:
len(train_dataset)

201

In [409]:
X_final = []

with torch.no_grad():
    for imgs, _ in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        X_final.append(outputs.cpu().numpy())

X_final = np.concatenate(X_final, axis=0)
X_final.shape

(2000, 1000)

In [410]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_final = scaler.transform(X_final)

In [474]:
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from pyod.models.iforest import IForest
from pyod.models.ocsvm import OCSVM
from pyod.models.knn import KNN
from pyod.models.abod import ABOD

model = ABOD(contamination=0.5)
model.fit(X)
y_preds = model.predict(X_final)

In [475]:
import os

def get_id(file_name):
    return int(str(file_name).split(".")[0].split("_")[1])

files = sorted(os.listdir("test_data/class_1"))
ids = [get_id(file) for file in files]
ids = sorted(ids)

ids[:20]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [476]:
y_preds

array([0, 1, 0, ..., 0, 1, 0])

In [477]:
import pandas as pd

submission = pd.DataFrame({
    "datapointID": ids,
    "subtaskID": 1,
    "answer": y_preds
})
submission.answer = submission.answer.map({0: 1, 1: 0})

submission

,datapointID,subtaskID,answer
0,0,1,1
1,1,1,0
2,2,1,1
3,3,1,1
4,4,1,0
...,...,...,...
1995,1995,1,1
1996,1996,1,0
1997,1997,1,1
1998,1998,1,0


In [478]:
submission.answer.value_counts()

answer
0    1045
1     955
Name: count, dtype: int64

In [479]:
submission.to_csv("submission.csv", index=False)